<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/1_bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Bit LLM

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
class BitLinear(nn.Linear):
  def forward(self, x: torch.Tensor):
    # Scale input tensor down to 8 bits
    i_scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    x_quant = torch.round(x * 127 / i_scale).clamp(-128, 127)

    # Scale layer weights to ternary -,+,0
    w = self.weight
    w_scale = w.abs().mean()
    w_quant = torch.round(w / (w_scale + 1e-5)).clamp(-1, 1)
    w_ternary = w + (w_quant - w).detach()

    out_t = F.linear(x_quant, w_ternary, self.bias)

    # Scale output tensor up to 8bits
    out_t = out_t * (127 / i_scale) * (1 / w_scale)
    return out_t